In [ ]:
## Heatmaps for min(B) and max(B) over a and delta_1
# y-axis: a in [100, 2000]
# x-axis: delta_1 in [0.01, 0.1]

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Fixed parameters
phi_B = 0.018
phi_H = 0.007
epsilon_B = 2000
b = 500
k = 5000
tau = 12
gamma_B = 0.003
gamma_H = 0.05
delta_2 = 0.06
epsilon_T = 1
gamma_T = 1 / 24
chi = 600
sigma = 300
theta = 105
kappa = 1 / 9

# Initial conditions
B0 = 10000
H0 = 20000
f0 = 1000
T0 = 2
V0 = 1

# Simulation setup
STARTTIME = 0
STOPTIME = 365
DT = 0.02
eps = 1e-6
lam = 120


def build_and_run(delta_1_val, a_val):
    q0 = delta_1_val * T0 / (a_val + B0) + delta_2 * V0 / (a_val + B0)
    I0 = q0 * tau

    def pos(x):
        return (x + Abs(x)) / 2

    IDX_B = 0
    IDX_H = 1
    IDX_T = 2
    IDX_V = 3
    IDX_f = 4
    IDX_I = 5

    B = y(IDX_B)
    H = y(IDX_H)
    Tm = y(IDX_T)
    V = y(IDX_V)
    f = y(IDX_f)
    I = y(IDX_I)

    B_tau = y(IDX_B, t - tau)
    T_tau = y(IDX_T, t - tau)
    V_tau = y(IDX_V, t - tau)

    B_pos = pos(B)
    H_pos = pos(H)
    T_pos = pos(Tm)
    V_pos = pos(V)
    f_pos = pos(f)
    I_pos = pos(I)
    B_tau_pos = pos(B_tau)
    T_tau_pos = pos(T_tau)
    V_tau_pos = pos(V_tau)

    S = (f_pos / (b + f_pos)) * (H_pos / (k + H_pos))
    infestation = delta_1_val * (B_pos / (a_val + B_pos)) * T_pos
    infection = delta_2 * (B_pos / (a_val + B_pos)) * V_pos

    q_now = delta_1_val * T_pos / (a_val + B_pos) + delta_2 * V_pos / (a_val + B_pos)
    q_tau = delta_1_val * T_tau_pos / (a_val + B_tau_pos) + delta_2 * V_tau_pos / (a_val + B_tau_pos)
    dI = q_now - q_tau

    exp_B = exp(-gamma_B * tau - I_pos)
    Phi = chi + sigma * cos(2 * pi * (t - theta) / 365)

    dB = epsilon_B * S - infestation - infection - gamma_B * B_pos - kappa * exp_B * B_tau_pos
    dH = kappa * exp_B * B_tau_pos - gamma_H * H_pos
    dT = epsilon_T * infestation - gamma_T * T_pos
    dV = epsilon_T * infection - gamma_T * V_pos
    df = Phi * (H_pos / (k + H_pos)) - phi_B * B_pos - phi_H * H_pos

    dB += lam * (B_pos - B)
    dH += lam * (H_pos - H)
    dT += lam * (T_pos - Tm)
    dV += lam * (V_pos - V)
    df += lam * (f_pos - f)
    dI += lam * (I_pos - I)

    dde = jitcdde([dB, dH, dT, dV, df, dI])

    def history(_s):
        return [B0, H0, T0, V0, f0, I0]

    dde.past_from_function(history)
    dde.step_on_discontinuities()
    dde.set_integration_parameters(
        atol=1e-8,
        rtol=1e-6,
        min_step=1e-12,
        max_step=DT,
        first_step=DT,
    )

    t0 = float(dde.t)
    start = max(STARTTIME + eps, t0)
    times = np.arange(start, STOPTIME + DT / 2, DT)

    minB = B0
    maxB = B0
    for tt in times:
        state = dde.integrate(float(tt))
        Bval = float(state[IDX_B])
        if Bval < minB:
            minB = Bval
        if Bval > maxB:
            maxB = Bval

    return minB, maxB


# Sweep ranges
N = 20
delta_1_vals = np.linspace(0.01, 0.1, N)
a_vals = np.linspace(100, 2000, N)

minB_map = np.zeros((len(a_vals), len(delta_1_vals)))
maxB_map = np.zeros((len(a_vals), len(delta_1_vals)))

for ia, a_val in enumerate(a_vals):
    for i1, d1 in enumerate(delta_1_vals):
        minB, maxB = build_and_run(d1, a_val)
        minB_map[ia, i1] = minB
        maxB_map[ia, i1] = maxB

In [ ]:
# Plot setup
base = plt.colormaps["viridis"]
viridis_light = LinearSegmentedColormap.from_list(
    "viridis_light", base(np.linspace(0.22, 1.00, 256))
)

na, n1 = minB_map.shape
d1_edges = np.linspace(delta_1_vals[0], delta_1_vals[-1], n1 + 1)
a_edges = np.linspace(a_vals[0], a_vals[-1], na + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)

# Left: min(B)
im0 = axes[0].pcolormesh(
    d1_edges, a_edges, minB_map,
    shading="flat", cmap=viridis_light,
    edgecolors=(1, 1, 1, 0.7), linewidth=0.6, antialiased=False
)
cbar0 = fig.colorbar(im0, ax=axes[0])
cbar0.set_label("Minimum brood population", fontsize=15)
cbar0.ax.tick_params(labelsize=18, width=1.6, length=6, colors="black")
cbar0.set_ticks([100, 5000, 10000])
cbar0.set_ticklabels(["100", "5,000", "10,000"])

#axes[0].set_title("Minimum $B$")
axes[0].set_xlabel("$\\delta_1$", fontsize=18)
axes[0].set_ylabel("$a$", fontsize=18)
axes[0].set_xlim(delta_1_vals[0], delta_1_vals[-1])
axes[0].set_ylim(a_vals[0], a_vals[-1])
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)
axes[0].tick_params(axis="both", labelsize=18)
axes[0].set_xticks([0.01, 0.065, 0.1])
axes[0].set_xticklabels(["0.01", "0.065", "0.1"])
axes[0].set_yticks([100, 2000])
axes[0].set_yticklabels(["100", "2,000"])
axes[0].text(-0.12, 1.15, "(a)", transform=axes[0].transAxes, color='black', fontsize=20, va='top', ha='left')
# Right: max(B)
im1 = axes[1].pcolormesh(
    d1_edges, a_edges, maxB_map,
    shading="flat", cmap=viridis_light,
    edgecolors=(1, 1, 1, 0.7), linewidth=0.6, antialiased=False
)
cbar1 = fig.colorbar(im1, ax=axes[1])
cbar1.set_label("Maximum brood population", fontsize=15)
cbar1.ax.tick_params(labelsize=18, width=1.6, length=6, colors="black")
cbar1.set_ticks([15622, 15715])
cbar1.set_ticklabels(["15,622", "15,715"])

#axes[1].set_title("Maximum $B$")
axes[1].set_xlabel("$\\delta_1$", fontsize=18)
axes[1].set_ylabel("$a$", fontsize=18)
axes[1].set_xlim(delta_1_vals[0], delta_1_vals[-1])
axes[1].set_ylim(a_vals[0], a_vals[-1])
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].tick_params(axis="both", labelsize=18)
axes[1].set_xticks([0.01, 0.065, 0.1])
axes[1].set_xticklabels(["0.01", "0.065", "0.1"])
axes[1].set_yticks([100, 2000])
axes[1].set_yticklabels(["100", "2,000"])

axes[1].text(-0.12, 1.15, "(b)", transform=axes[1].transAxes, color='black', fontsize=20, va='top', ha='left')
plt.savefig("heatmap_2.png", dpi=300, bbox_inches="tight")
plt.show()